# UZK: S4-автоэнкодер

In [ ]:
import os
from pathlib import Path

# Resolve notebook paths from the project root, including nested notebook directories.
working_dir = Path.cwd().resolve()
project_root = next(
    (
        directory
        for directory in (working_dir, *working_dir.parents)
        if (directory / "pyproject.toml").is_file()
        and (directory / "src/feature_selector").is_dir()
    ),
    None,
)
if project_root is None:
    raise RuntimeError("Open the notebook from within the feature_selector project")
os.chdir(project_root)


In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display

from feature_selector import ExperimentRun, load_config, run_experiment
from feature_selector.artifacts import load_run
from feature_selector.data import load_features
from feature_selector.evaluation import report_metrics
from feature_selector.pipeline.data import prepare_data
from feature_selector.visualization import (
    plot_reconstruction_predictions,
    plot_training_history,
)

config = load_config(Path("configs/uzk/s4_autoencoder/pretrain.yaml"))
data = load_features(config.data["features_path"], config.data["freq"])


## Признаки для графиков

In [ ]:
plot_features = [
    "81TI10143",
    "81TI10126",
    "81FIL30066",
    "81LILH40012",
    "81TIH11209",
    "81FCL30063",
    "81FI30052",
    "81TI10123",
]
missing = [feature for feature in plot_features if feature not in data.X.columns]
if missing:
    raise ValueError(f"Missing plot features: {missing}")
print(f"Training features: {data.X.shape[1]}; plotted features: {len(plot_features)}")


## Обучение

In [ ]:
run = run_experiment(config)
assert isinstance(run, ExperimentRun)
print(f"Результаты: {run.artifact_dir}")
pd.DataFrame.from_dict(
    {split: metrics["overall"] for split, metrics in run.metrics.items()},
    orient="index",
)


## Экспорт кодировщика для ансамбля

In [ ]:
encoder_path = run.pipeline.export_transform(
    Path("runs/uzk_s4_encoder.pkl"), through="autoencoder", mode="encode"
)
print(f"Saved encoder pipeline: {encoder_path}")


## История обучения

In [ ]:
ax = plot_training_history(run.model.history_, run.model.best_epoch_)
ax.set_ylabel("MSE (model input units)")
plt.show()

pd.DataFrame.from_dict(
    run.metrics["validation"]["per_feature"], orient="index"
).loc[plot_features]


## Реконструкция восьми выбранных признаков

In [ ]:
reports = dict(run.predictions)
fig, axes = plt.subplots(4, 2, figsize=(18, 16), sharex=True, layout="constrained")
for feature, ax in zip(plot_features, axes.flat):
    plot_reconstruction_predictions(reports, feature, ax=ax)
plt.show()


## Латентное представление

In [ ]:
prepared = prepare_data(data, run.boundaries, config.data["freq"])
windows = run.pipeline.transform(prepared, through="windows").split("validation")
encoded = run.model.encode(windows.X)
reconstructed = run.model.decode(encoded)
print("Input:", windows.X.shape)
print("Encoded:", encoded.shape)
print("Reconstructed:", reconstructed.shape)


## Загрузка и явная оценка test


In [ ]:
saved = load_run(run.artifact_dir)
assert isinstance(saved, ExperimentRun)
test_report = saved.predict(data, split="test")
print(f"Test feature-point pairs: {len(test_report)}")
display(pd.DataFrame.from_dict(report_metrics(test_report)["per_feature"], orient="index")
        .loc[plot_features])


In [ ]:
batch_history = pd.DataFrame([
    {"epoch": epoch["epoch"], **batch}
    for epoch in run.model.history_ for batch in epoch["batches"]
])
display(batch_history.head())
if not batch_history.empty:
    batch_history.plot(x="step", y="train_loss", title="Training batch loss")
    plt.show()
